### Positional Embedding, how words are represented in the llm

As we said embeddings are multi dimensional rapresentation of the tokens; what really a token represent is basically unknow especially byte pair encoding or character level encoding, like we explained before.

without positional embedding this two phrases would mean the same:
- the cat bites the dog
- the dog bites the cat 

So what we do? to each embedded word we add a positional vector to represent the word in the phrase:
There are some type of positional encoding we may use:

### 1. Sinosuoidal Embedding

In the original Transformer (“Attention Is All You Need”), positional encodings are fixed, not learned: they are deterministic functions of the position using sines and cosines at different frequencies.

$$
\begin{aligned}
PE_{(pos, 2i)} &= \sin\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right) \\
PE_{(pos, 2i+1)} &= \cos\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)
\end{aligned}
$$

- 𝑝𝑜𝑠 : token index in the sequence (0, 1, 2, …).
- i: dimension index (0, 1, …) 
- 𝑑model : il numero di dimensioni di questo vettore.
- Odd dimensions: use cosine,  Even dimensions: use sine.
    - Component 0 → sine
    - Component 1 → cosine
    - Component 2 → sine
    - Component 3 → cosine
- 10000: a scaling base that spreads frequencies across dimensions


In [ ]:
import numpy as np

def sinosuoidal_positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model)) # our positional embedder
    
    # Ciclo 1: Per ogni posizione "pos" nella frase (0, 1, 2...)
    for pos in range(max_len):
        
        # Ciclo 2: Per ogni dimensione "j" del vettore (0, 1, ..., 511)
        for j in range(d_model):
            
            # Qui applichiamo la logica: "j" corrisponde a "2i" o "2i+1"
            # Quindi "i" è la parte intera di j diviso 2
            i = j // 2
            
            # Calcoliamo il denominatore della formula: 10000^(2i / d_model)
            denominatore = 10000 ** ((2 * i) / d_model)
            
            if j % 2 == 0:
                # Se l'indice j è PARI -> usa SENO
                PE[pos, j] = np.sin(pos / denominatore)
            else:
                # Se l'indice j è DISPARI -> usa COSENO
                PE[pos, j] = np.cos(pos / denominatore)
                
    return PE

# --- TESTIAMOLO ---
d_model = 6  # Teniamo numeri piccoli per leggere l'output
max_len = 4
pe_matrix = positional_encoding_letterale(max_len, d_model)

print("Matrice PE (arrotondata per leggere meglio):\n")
print(np.round(pe_matrix, 2))

### 2. Learned Absolute Embedding

In the original Transformer paper (*Attention Is All You Need*), positional information was injected using fixed mathematical formulas (Sine/Cosine).

Learned Absolute Positional Embedding is the alternative approach popularized by models like BERT (Google) and GPT-2 (OpenAI). Instead of calculating positions using trigonometry, the model learns the optimal vector for every position from scratch during the training phase.

The model allocates a specific Lookup Table (a matrix of learnable parameters) dedicated solely to positions and for each position of each token in the sentence there's a dedicated positional embedding
let's imagine this better with an example:

- the cat is on the table:

word|token|embedding|positonal embedding for the position

the -> 1 -> [2,2,9..] + [3,2,5,2...]  
cat -> 2 -> [23,7,3..] +[...]  
is  -> 3 -> [...] +[...]  
on  -> 4 -> [...] +[...]  
the -> 5 -> [...] +[...]  
table -> 6 -> [2,4,12,4..] + +[...]  

This is the working of the positional embedding is importat to notice that the positional weights gets
trained as weights by the magic of backpropagation. 
Of course this approach has some defects since is prone to overfitting and underfitting, and lack of form variance.

#### **What do the weights actually learn? (Soft partitioning)**

You might wonder: *Doesn't adding a position vector to the word vector destroy the word's meaning?*

In high-dimensional spaces, the model instead learns to organize information via **soft partitioning**. Through backpropagation, the positional weights tend to become almost “orthogonal” to the semantic weights.

- **Semantic dimensions:**  
  The word embeddings use most dimensions (e.g., indices 0–600) to store meaning (e.g., “cat” vs “dog”).

- **Positional dimensions:**  
  The positional embeddings learn to activate mainly in the remaining dimensions (e.g., indices 601–768), keeping their values small in the semantic dimensions.

This allows the sum 

#### **Critical limitation**

Despite being the standard for BERT-era models, this approach introduces specific biases.

- Positional overfitting (spurious correlations) : The model learns not just *where* a token is, but    *what usually happens there*.

- **Scenario:**  
  In Wikipedia training data, in position 0 is almost always a subject or article (“The”, “He”, “A”, “It”).

- **Result:**  
  The vector for position 0 learns a bias toward **noun-like** behavior.  
  If, during inference, you provide an imperative sentence starting with a verb (e.g., “Run!”), the positional embedding for Pos(0) conflicts with the semantic embedding of “Run”, potentially causing errors.

#### The "long tail" underfitting

Not all positions are created equal.

- **Pos 0–10:** Updated in almost every training step → high-quality, well-trained vectors.  
- **Pos 500–512:** Updated only when the batch contains a max-length sequence → rare updates.

**Result:**  
The model tends to be “smarter” at the beginning of sentences and “dumber” (noisier) at the extreme end of the context window.

#### Lack of translation invariance

This is the most significant theoretical flaw.

- **Sentence A:** “The cat eats” → “cat” at Pos\(_1\)  
- **Sentence B:** “Today, the cat eats” → “cat” at Pos\(_3\)

Since position 1 and 2 are independent learnable vectors, the model does not natively understand that “cat” is the same entity just shifted by two steps. It effectively has to **relearn “cat” in each possible position**.

This limitation is a major reason why modern LLMs (e.g., Llama 3, Mistral) have moved to **RoPE (Rotary Positional Embeddings)**, which address translation invariance using relative rotations in embedding space.

### 3. Rotary Positional Embeddings (RoPE)

Rotary Positional Embeddings were introduced to fix the main weaknesses of learned absolute embeddings, especially the **lack of translation invariance** and poor behavior at unseen lengths.

Instead of **adding** a positional vector to the token embedding, RoPE **rotates** the token representation in a position-dependent way.  
The key idea: *position = rotation in a shared space*, not a separate vector that gets added.

#### Intuition: rotate, don’t add

With RoPE, each token embedding still encodes its meaning (like “cat”, “dog”, “run”), but its **position** is encoded as a **rotation angle** applied to that embedding.

- For each pair of embedding dimensions \((2i, 2i+1)\), we treat them like a 2D plane.
- Position \(p\) corresponds to a rotation by some angle \(\theta_{p,i}\) in that 2D plane.
- Higher dimensions use different frequencies (different rotation speeds), similar to sinusoidal encodings.

The **direction** of the vector changes with position, but its **content** remains structured.

#### How RoPE is applied in practice

RoPE is not applied directly to the raw token embeddings in most implementations.  
Instead, it is applied to the **query** and **key** vectors inside the self-attention mechanism.

1. Start from token embeddings → project to queries and keys:
2. Split into 2D pairs:
3. For each pair and for each position \(p\), apply a rotation:

The angle is usually defined using sinusoidal frequencies (like the original Transformer), so RoPE is a **rotational reinterpretation** of sinusoidal encoding.


#### Why RoPE fixes translation invariance

Consider the two sentences:

- Sentence A: “The cat eats” → “cat” at position 1  
- Sentence B: “Today, the cat eats” → “cat” at position 3  

With learned absolute embeddings, position 1 and position 3 have **completely unrelated vectors**, so the model must relearn “cat at pos 1”, “cat at pos 2”, “cat at pos 3”, etc.

With RoPE:

- The query/key for “cat at position 3” is just the **query/key for “cat at position 1” rotated by the extra offset**.
- Relative distances are preserved as differences in rotation angles.

This means:

- “cat at pos 3” attending to “eats at pos 4” has the **same relative geometry** as  
  “cat at pos 1” attending to “eats at pos 2”.

The attention scores become naturally **relative-position aware**, without needing an explicit relative position bias table.


#### Benefits over learned absolute embeddings

RoPE brings several important advantages:

- **Built-in relative position awareness**  
  Attention depends on relative rotation, so shifting the whole sentence by a few positions does not fundamentally change the pattern.

- **Better generalization to longer sequences**  
  Because rotations are defined analytically (via sin/cos frequencies), RoPE can be extended beyond the context lengths seen during training (up to some practical limit).

- **No separate lookup table for positions**  
  There is no positional lookup table that overfits to specific absolute positions or suffers from long-tail undertraining (pos 500–512).

- **More parameter-efficient and structured**  
  Position is encoded in a mathematically constrained way (rotations), not as arbitrary per-position vectors.

#### Why modern LLMs use RoPE

Models like **Llama 2/3**, **Mistral**, and many recent architectures adopt RoPE (often with some scaling tricks) because it:

- avoids positional overfitting to specific absolute locations,
- handles long contexts more gracefully,
- and provides a cleaner, more theoretically grounded notion of “position as transformation”, not “position as another embedding vector”.

In short:  
**Learned absolute embeddings say:** “position 17 has its own vector.”  
**RoPE says:** “moving everything by +k just rotates query/key vectors accordingly, preserving relational structure.”
